In [20]:
import openai
import requests
import json

client = openai.OpenAI()

messages = []

In [21]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description":"https://nomad-movies-2.nomadcoders.workers.dev/movies API 호출을 통해 인기 영화를 가져옵니다.",
            "parameters":{
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description":"https://nomad-movies-2.nomadcoders.workers.dev/{id} API 호출을 통해 영화 정보를 가져옵니다.",
            "parameters":{
                "type": "object",
                "properties": {
                    "id" : {
                        "type": "integer",
                        "description": "영화의 id"
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description":"https://nomad-movies-2.nomadcoders.workers.dev/{id}/credits API 호출을 통해 영화 출연진 및 제작진 정보를 가져옵니다.",
            "parameters":{
                "type": "object",
                "properties": {
                    "id" : {
                        "type": "integer",
                        "description": "영화의 id"
                    }
                },
                "required": ["id"]
            }
        }
    }
]

In [22]:
def get_popular_movies():
    response = requests.get("https://nomad-movies-2.nomadcoders.workers.dev/movies")
    return response.json()

def get_movie_details(id):
    response = requests.get(f"https://nomad-movies-2.nomadcoders.workers.dev/movies/{id}")
    return response.json()

def get_movie_credits(id):
    response = requests.get(f"https://nomad-movies-2.nomadcoders.workers.dev/movies/{id}/credits")
    return response.json()

FUNCTION_MAP = {
    'get_popular_movies': get_popular_movies,
    'get_movie_details': get_movie_details,
    'get_movie_credits': get_movie_credits
}

In [23]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments
                        }
                    } for tool_call in message.tool_calls
                ]
        })

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": json.dumps(result)
            })

        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, tools=TOOLS
    )
    process_ai_response(response.choices[0].message)

In [24]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({
            "role": "user",
            "content": message
        })
        print(f"User: {message}")
        call_ai()

User: 지금 인기 있는 영화가 무엇인지 알려줘
AI: 지금 인기 있는 영화는 다음과 같습니다:

1. **Obsession**
   - **개요**: 신비로운 "원 위시 윌로우"를 깨트린 후, 주인공은 자신의 사랑을 얻기 위해 원하는 것을 가지게 되지만, 그 욕망이 어두운 대가를 초래하게 됨을 깨닫게 된다.
   - **개봉일**: 2026-05-13
   - **평점**: 7.9
   - ![Obsession 포스터](https://image.tmdb.org/t/p/w780/2G249T4Sgu8gXIZpaXWnxHYYNQV.jpg)

2. **Peddi**
   - **개요**: 1980년대 아나드라 프라데시의 한 마을에서 열정적인 한 주민이 스포츠를 통해 공동체를 단결시켜 강력한 라이벌에 맞서 자존심을 지키기 위해 싸운다.
   - **개봉일**: 2026-06-03
   - **평점**: 6.3
   - ![Peddi 포스터](https://image.tmdb.org/t/p/w780/kJAJNNBYlbqAcpTDxBNnaILSMTy.jpg)

3. **Lee Cronin's The Mummy**
   - **개요**: 기자의 어린 딸이 사막에서 실종된 지 8년이 지나 그녀가 돌아오지만, 기쁨의 재회의 순간이 악몽으로 바뀌게 된다.
   - **개봉일**: 2026-04-15
   - **평점**: 8.1
   - ![The Mummy 포스터](https://image.tmdb.org/t/p/w780/1q308iixueCU4pFtSYugNOevtNx.jpg)

4. **The Mandalorian and Grogu**
   - **개요**: 악의 제국이 무너진 이후, 새로운 공화국이 전투를 준비하며 전설적인 맨달로리안 바운티 헌터와 그의 제자가 힘을 합친다.
   - **개봉일**: 2026-05-20
   - **평점**: 6.8
   - ![Mandalorian 포스터](https://image.tmdb.org/t/p/w780/5Vi8dSauVwH1